### Escalar los datos usando StandardScaler() o MinMaxScaler(), guardar el objeto escalador en un archivo pickle.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import pickle
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans


# Modelos
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor


# Métricas
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
df = pd.read_csv("madrid_sale_properties_cleaned.csv")

df

,price_eur,barrio,distrito,latitude,longitude,energy_consumption_rating,energy_consumption_value,energy_emissions_rating,energy_emissions_value,adaptado a personas con movilidad reducida,...,piscina,planta,portero automático,puerta blindada,sistema de seguridad,superficie construida,superficie útil,terraza,trastero,vidrios dobles
0,9500000.0,Recoletos,Salamanca,40.422116,-3.683149,False,NaN,False,NaN,0.0,...,False,1.0,Portero físico,False,False,499.0,499.0,False,False,False
1,840000.0,Universidad-Malasaña,Centro,40.422312,-3.706137,False,NaN,False,NaN,0.0,...,False,2.0,True,False,False,109.0,95.0,False,False,False
2,3600000.0,Goya,Salamanca,40.424364,-3.670577,False,NaN,False,NaN,0.0,...,False,3.0,True,True,False,379.0,379.0,False,False,False
3,285000.0,Opañel,Carabanchel,40.389100,-3.719900,E,151 kWh/m² año,D,31 Kg CO₂/m² año,0.0,...,False,0.0,False,False,False,60.0,60.0,False,False,False
4,539000.0,Rejas,San Blas,40.442042,-3.575920,E,152 kWh/m² año,D,30 Kg CO₂/m² año,1.0,...,True,2.0,True,True,False,127.0,101.0,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2997,279000.0,Portazgo,Puente de Vallecas,40.383712,-3.650452,E,211 kWh/m² año,E,44 Kg CO₂/m² año,0.0,...,False,2.0,False,False,False,113.0,113.0,False,False,False
2998,411000.0,Pueblo Nuevo,Ciudad Lineal,40.426333,-3.638205,False,NaN,False,NaN,0.0,...,False,0.0,False,False,False,130.0,120.0,False,False,False
2999,1550000.0,Valdemarín,Moncloa-Aravaca,40.465343,-3.762967,D,132 kWh/m² año,D,27 Kg CO₂/m² año,0.0,...,True,2.0,False,False,False,239.0,239.0,False,False,False
3000,2490000.0,Trafalgar,Chamberí,40.435597,-3.700479,False,NaN,False,NaN,0.0,...,False,4.0,False,True,False,185.0,160.0,False,False,False


In [3]:
# Mostrar valores nulos ordenados (descendente)
df.isnull().sum().sort_values(ascending=False).head(8)

energy_consumption_value    2327
energy_emissions_value      2327
antigüedad                  1311
superficie útil               10
superficie construida         10
habitaciones                   7
distrito                       1
barrio                         1
dtype: int64

In [4]:
# Eliminar columnas que no aportan información al modelo de machine learning, o tienen muchos NaN para poder imputarlos.
df = df.drop(['energy_consumption_value', 'energy_emissions_value', 'antigüedad'], axis=1)

# Eliminar filas donde 'barrio' o 'distrito' son NaN
df = df.dropna(subset=['barrio', 'distrito'])

In [5]:
# Imputar con KNN los NaN en las columnas superficie construida, superficie útil y habitaciones
columnas_a_imputar = ['superficie útil', 'superficie construida', 'habitaciones']

# Asegurar que las columnas están en formato numérico
df[columnas_a_imputar] = df[columnas_a_imputar].apply(pd.to_numeric, errors='coerce')

# Crear una copia del DataFrame con solo las columnas necesarias para la imputación
df_imputacion = df[columnas_a_imputar]

# Inicializar el imputador con K vecinos
imputer = KNNImputer(n_neighbors=5)
valores_imputados = imputer.fit_transform(df_imputacion)

# Reemplazar en el DataFrame original
df[columnas_a_imputar] = valores_imputados


In [6]:
# Encoding. Pasar columnas categóricas a numéricas

# Identificar columnas categóricas
cat_columns = df.select_dtypes(include=['object', 'bool']).columns

# Usar One-Hot Encoding
df_encoded = pd.get_dummies(df, columns=cat_columns, drop_first=True)


### Definir los conjuntos de Train y Test

In [7]:

X = df_encoded.drop("price_eur", axis=1)
y = df_encoded["price_eur"]

print(f"X: {X.shape}")
print(f"y: {y.shape}")

X: (3001, 212)
y: (3001,)


In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)


print(f"Conjunto de Train: {X_train.shape, y_train.shape}")
print(f"Conjunto de Test: {X_test.shape, y_test.shape}")

Conjunto de Train: ((2400, 212), (2400,))
Conjunto de Test: ((601, 212), (601,))


In [9]:
# Escalar los datos ya codificados con MinMaxScaler
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Guardar el escalador
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)


### Define un modelo de regresión que pueda predecir el precio de un piso/casa según sus características dependiendo del clustering que pertenecen.

In [10]:
# Determinar número de clústeres 

kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(X_scaled)
df_encoded["cluster"] = clusters


In [11]:
# Comparar varios modelos por cluster para definir el modelo más óptimo

resultados_modelos = []

# Diccionario de modelos a comparar
modelos_disponibles = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "KNN": KNeighborsRegressor(),
    "SVR": SVR()
}

for cluster_id in np.unique(clusters):
    print(f"\n🔹 Comparando modelos para clúster {cluster_id}...")

    # Filtrar datos por clúster
    df_cluster = df_encoded[df_encoded["cluster"] == cluster_id]
    X_c = df_cluster.drop(["price_eur", "cluster"], axis=1)
    y_c = df_cluster["price_eur"].values.reshape(-1, 1)

    # Dividir datos
    X_train, X_test, y_train, y_test = train_test_split(X_c, y_c, test_size=0.2, random_state=42)


    # Escalar características
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled = scaler_X.transform(X_test)
    y_train_scaled = scaler_y.fit_transform(y_train)
    y_test_scaled = scaler_y.transform(y_test)

    # Guardar scaler_X
    with open('scaler_X.pkl', 'wb') as f:
        pickle.dump(scaler_X, f)

    # Guardar scaler_y
    with open('scaler_y.pkl', 'wb') as f:
        pickle.dump(scaler_y, f)

    for nombre_modelo, modelo in modelos_disponibles.items():
        # Entrenar
        modelo.fit(X_train_scaled, y_train_scaled.ravel())

        # Predecir y desescalar
        y_pred_scaled = modelo.predict(X_test_scaled).reshape(-1, 1)
        y_pred = scaler_y.inverse_transform(y_pred_scaled)
        y_test_inv = scaler_y.inverse_transform(y_test_scaled)

        # Calcular métricas
        mae = mean_absolute_error(y_test_inv, y_pred)
        mse = mean_squared_error(y_test_inv, y_pred)
        r2 = r2_score(y_test_inv, y_pred)

        print(f"  {nombre_modelo:>15} → R²: {r2:.3f} | MAE: {mae:.1f} | MSE: {mse:.1f}")

        # Guardar resultados
        resultados_modelos.append({
            "Cluster": cluster_id,
            "Modelo": nombre_modelo,
            "MAE": mae,
            "MSE": mse,
            "R2": r2,
            "Train Size": len(X_train),
            "Test Size": len(X_test)
        })

# Convertir resultados a DataFrame y guardar
df_resultados_modelos = pd.DataFrame(resultados_modelos)
df_resultados_modelos = df_resultados_modelos.sort_values(by=["Cluster", "R2"], ascending=[True, False])

# Guardar resultados para Streamlit
df_resultados_modelos.to_csv("comparativa_modelos_por_cluster.csv", index=False)

# Mostrar tabla
df_resultados_modelos



🔹 Comparando modelos para clúster 0...
  Linear Regression → R²: 0.656 | MAE: 407551.6 | MSE: 701499407217.6
    Decision Tree → R²: 0.468 | MAE: 374768.0 | MSE: 1083630783319.1
    Random Forest → R²: 0.755 | MAE: 253209.1 | MSE: 500069404536.1
              KNN → R²: 0.474 | MAE: 448504.9 | MSE: 1071116715425.3
              SVR → R²: 0.532 | MAE: 704769.1 | MSE: 954073295345.9

🔹 Comparando modelos para clúster 1...
  Linear Regression → R²: 0.723 | MAE: 543443.0 | MSE: 797588907743.1
    Decision Tree → R²: 0.690 | MAE: 432195.5 | MSE: 894247543813.4
    Random Forest → R²: 0.843 | MAE: 328246.9 | MSE: 453878485885.5
              KNN → R²: 0.570 | MAE: 616425.2 | MSE: 1240151014384.2
              SVR → R²: 0.680 | MAE: 740279.9 | MSE: 921790677468.5

🔹 Comparando modelos para clúster 2...
  Linear Regression → R²: 0.652 | MAE: 387976.3 | MSE: 384467580995.4
    Decision Tree → R²: 0.792 | MAE: 239762.0 | MSE: 229862430871.8
    Random Forest → R²: 0.781 | MAE: 231844.5 | MSE: 24

,Cluster,Modelo,MAE,MSE,R2,Train Size,Test Size
2,0,Random Forest,253209.064587,5.000694e+11,0.754585,1498,375
0,0,Linear Regression,407551.568844,7.014994e+11,0.655731,1498,375
4,0,SVR,704769.116537,9.540733e+11,0.531778,1498,375
3,0,KNN,448504.877867,1.071117e+12,0.474338,1498,375
1,0,Decision Tree,374768.026667,1.083631e+12,0.468196,1498,375
7,1,Random Forest,328246.875588,4.538785e+11,0.842636,408,102
5,1,Linear Regression,543442.975242,7.975889e+11,0.723469,408,102
6,1,Decision Tree,432195.509804,8.942475e+11,0.689956,408,102
9,1,SVR,740279.850747,9.217907e+11,0.680407,408,102
8,1,KNN,616425.237255,1.240151e+12,0.570028,408,102


In [12]:
# Entrenamos el modelo más optimo. RandomForestRegressor

resultados_modelo_optimo = []
modelos = {}

for cluster_id in np.unique(clusters):
    print(f"Entrenando modelo para clúster {cluster_id}...")

    df_cluster = df_encoded[df_encoded["cluster"] == cluster_id]
    X_c = df_cluster.drop(["price_eur", "cluster"], axis=1)
    y_c = df_cluster["price_eur"]

    X_train, X_test, y_train, y_test = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

    modelo = RandomForestRegressor(random_state=42)
    modelo.fit(X_train, y_train)

    y_pred = modelo.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"Clúster {cluster_id} → MSE: {mse:.2f} | MAE: {mae:.2f} | R²: {r2:.4f}")

    # Guardar resultado
    resultados_modelo_optimo.append({
        "Cluster": cluster_id,
        "MSE": mse,
        "MAE": mae,
        "R²": r2,
        "Train Size": len(X_train),
        "Test Size": len(X_test)
    })

    modelos[cluster_id] = modelo

# Convertir resultados a DataFrame y guardar
df_resultados = pd.DataFrame(resultados_modelos)
df_resultados.to_csv("resultados_modelo_optimo.csv", index=False)

# Guardar modelo y objetos necesarios
with open("modelo_optimo_por_cluster.pkl", "wb") as f:
    pickle.dump(modelos, f)

with open("kmeans_model.pkl", "wb") as f:
    pickle.dump(kmeans, f)

with open("X_columns.pkl", "wb") as f:
    pickle.dump(X.columns.tolist(), f)


Entrenando modelo para clúster 0...


Clúster 0 → MSE: 505223868361.41 | MAE: 254288.65 | R²: 0.7521
Entrenando modelo para clúster 1...
Clúster 1 → MSE: 476728968090.36 | MAE: 332018.72 | R²: 0.8347
Entrenando modelo para clúster 2...
Clúster 2 → MSE: 233646198744.72 | MAE: 228945.94 | R²: 0.7887


In [15]:
# Función de predicción
def predecir_precio(nueva_vivienda):
    # Asegurarse de tener las mismas columnas
    df_nuevo = pd.DataFrame([nueva_vivienda])
    df_nuevo_encoded = pd.get_dummies(df_nuevo, drop_first=True)

    # Cargar columnas originales
    with open("X_columns.pkl", "rb") as f:
        columnas_originales = pickle.load(f)

    # Agregar columnas faltantes y ordenar
    for col in columnas_originales:
        if col not in df_nuevo_encoded.columns:
            df_nuevo_encoded[col] = 0
    df_nuevo_encoded = df_nuevo_encoded[columnas_originales]

    # Escalar
    with open("scaler_X.pkl", "rb") as f:
        scaler = pickle.load(f)
    X_nuevo_scaled = scaler.transform(df_nuevo_encoded)

    # Determinar clúster
    with open("kmeans_model.pkl", "rb") as f:
        kmeans = pickle.load(f)
    cluster = kmeans.predict(X_nuevo_scaled)[0]

    # Predecir
    with open("modelos_por_cluster.pkl", "rb") as f:
        modelos = pickle.load(f)
    pred = modelos[cluster].predict(df_nuevo_encoded)[0]
    
    return cluster, pred

In [16]:
# Ejemplo para comprobar función predecir_precio

nueva_vivienda = {
    "superficie útil": 80,
    "superficie construida": 90,
    "baños": 2,
    "distrito": "Centro",
    "habitaciones": 3,
    "barrio": "Sol",
    "planta": "3",
    "exterior": "True",
    "antigüedad": "Entre 10 y 20 años",
    "terraza": "False",
    "garaje": "True"
}

cluster_pred, precio_pred = predecir_precio(nueva_vivienda)

print(f"Clúster asignado: {cluster_pred}")
print(f"Precio estimado: {precio_pred:.2f} EUR")


C:\Users\Marta\AppData\Local\Temp\ipykernel_7860\328443691.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_nuevo_encoded[col] = 0
C:\Users\Marta\AppData\Local\Temp\ipykernel_7860\328443691.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_nuevo_encoded[col] = 0
C:\Users\Marta\AppData\Local\Temp\ipykernel_7860\328443691.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd

Clúster asignado: 1
Precio estimado: 369933.80 EUR


### Puedes usar también modelos de Redes Neuronales. 

### Guarda en una tabla los resultados de los modelos para mostrarlos en Streamlit.

In [ ]:
# Código para guardar en Streamlit

import streamlit as st
import pandas as pd

df_resultados = pd.read_csv("resultados_modelos.csv")

st.title("Evaluación de Modelos por Clúster")
st.dataframe(df_resultados.style.format({
    "MSE": "{:,.2f}",
    "MAE": "{:,.2f}",
    "R²": "{:.4f}"
}))